In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:

# ==========================================
# PATH MODEL (ISI SESUAI PUNYA KAMU)
# ==========================================
MODEL_PATHS = {
    "Model-RDC-1.1": "../models/Model-RDC-1.1/model.h5",
    "Model-RDC-2.1": "../models/Model-RDC-2.1/model.h5",
    "Model-RDC-3.1": "../models/Model-RDC-3.1/model.h5",
    "Model-RDC-4.1": "../models/Model-RDC-4.1/model.h5",
}


In [ ]:
# ==========================================
# STEP 0: PARAMETERS & PATHS (sesuaikan)
# ==========================================
PROCESSED_DIR = "../dataset_split_0"  # output: train/ val/ test/ per kelas

IMG_SIZE = (224, 224)   # tuple: target_size untuk flow_from_directory
IMG_SIDE = IMG_SIZE[0]  # integer untuk fungsi make_square
BATCH_SIZE = 16

In [ ]:

# ==========================================
# STEP 4: Data generators
# - train: augmentation + rescale
# - val/test: hanya rescale
# ==========================================

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

valtest_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    os.path.join(PROCESSED_DIR, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
)

val_generator = valtest_datagen.flow_from_directory(
    os.path.join(PROCESSED_DIR, "val"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = valtest_datagen.flow_from_directory(
    os.path.join(PROCESSED_DIR, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


In [ ]:

# ==========================================
# FUNCTION EVALUASI
# ==========================================
def evaluate_model(model, generator):
    generator.reset()

    y_pred = model.predict(generator, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = generator.classes

    report = classification_report(
        y_true,
        y_pred_classes,
        target_names=list(generator.class_indices.keys()),
        output_dict=True,
        zero_division=0
    )

    return {
        "accuracy": report["accuracy"],
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1-score": report["weighted avg"]["f1-score"]
    }


In [ ]:

# ==========================================
# LOOP SEMUA MODEL
# ==========================================
results = []

for model_name, model_path in MODEL_PATHS.items():
    print(f"\n🚀 Evaluasi {model_name}...")

    model = load_model(model_path)

    metrics = evaluate_model(model, val_generator)

    metrics = {k: round(v, 2) for k, v in metrics.items()} 

    metrics["model"] = model_name
    results.append(metrics)


In [ ]:

# ==========================================
# BUAT DATAFRAME
# ==========================================
df_results = pd.DataFrame(results)

# urutkan kolom biar rapi
df_results = df_results[["model", "accuracy", "precision", "recall", "f1-score"]]

print("\n📊 Tabel Perbandingan Model:")
print(df_results)


In [ ]:

# ==========================================
# SIMPAN KE CSV
# ==========================================
# save_path = os.path.join(MODEL_DIR, "model_comparison.csv")
# df_results.to_csv(save_path, index=False)

# print(f"\n✅ Disimpan ke: {save_path}")